In [19]:
# ============================================================
# AUDIT DPO COMPLET — Chemins + chargement + contrôles
# ============================================================

import hashlib
import re
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd


# Localise la racine du projet et définit les chemins DPO.

def find_project_root(start: Path | None = None) -> Path:
    """Localise la racine du projet."""
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw" / "datasets").is_dir():
            return candidate

    raise FileNotFoundError("Répertoire data/raw/datasets introuvable.")


PROJECT_ROOT = find_project_root()

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "dpo_dataset_creation"
TRIAGE_DIR = OUTPUT_DIR / "priority_pools"
RUN_SPLIT_DIR = OUTPUT_DIR / "splits"
AUDIT_DIR = OUTPUT_DIR / "audit"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projet       : {PROJECT_ROOT}")
print(f"Pools triage : {TRIAGE_DIR}")
print(f"Splits       : {RUN_SPLIT_DIR}")
print(f"Audit        : {AUDIT_DIR}")


# Charge les trois pools DPO restructurés.

DPO_FILES = {
    "P1": TRIAGE_DIR / "dpo_p1_restructure_sft_aligned_p1_safety_corrected.jsonl",
    "P2": TRIAGE_DIR / "dpo_p2_restructure_sft_aligned_p1_safety_corrected.jsonl",
    "P3": TRIAGE_DIR / "dpo_p3_restructure_sft_aligned_p1_safety_corrected.jsonl",
}

missing_files = [str(path) for path in DPO_FILES.values() if not path.is_file()]

if missing_files:
    raise FileNotFoundError(
        "Fichiers DPO introuvables :\n" + "\n".join(missing_files)
    )

frames = []

for source, path in DPO_FILES.items():
    part = pd.read_json(path, lines=True)
    part["source_file"] = source
    frames.append(part)

df = pd.concat(frames, ignore_index=True)


# Vérifie le schéma attendu.

EXPECTED = [
    "id",
    "language",
    "source_split",
    "model_context",
    "chosen_explanation",
    "chosen_priority",
    "rejected_explanation",
    "rejected_priority",
    "context_hash",
    "source_file",
]

missing_columns = sorted(set(EXPECTED) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(EXPECTED))

if missing_columns:
    raise ValueError(f"Colonnes obligatoires absentes : {missing_columns}")

TEXT_COLS = [
    "model_context",
    "chosen_explanation",
    "rejected_explanation",
]

for col in TEXT_COLS:
    df[col] = df[col].fillna("").astype(str)

df["chosen_priority"] = pd.to_numeric(
    df["chosen_priority"], errors="coerce"
)
df["rejected_priority"] = pd.to_numeric(
    df["rejected_priority"], errors="coerce"
)


# Contrôle l'intégrité générale.

df["id_missing"] = (
    df["id"].isna()
    | df["id"].astype(str).str.strip().eq("")
)
df["duplicate_id"] = df["id"].duplicated(keep=False)

df["context_empty"] = df["model_context"].str.strip().eq("")
df["chosen_empty"] = df["chosen_explanation"].str.strip().eq("")
df["rejected_empty"] = df["rejected_explanation"].str.strip().eq("")

df["chosen_invalid"] = ~df["chosen_priority"].isin([1, 2, 3])
df["rejected_invalid"] = ~df["rejected_priority"].isin([1, 2, 3])
df["same_priority"] = df["chosen_priority"].eq(df["rejected_priority"])


# Vérifie les context_hash et les contextes dupliqués.

df["computed_hash"] = df["model_context"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()
)

df["hash_invalid"] = df["context_hash"].ne(df["computed_hash"])
df["duplicate_context"] = df["context_hash"].duplicated(keep=False)

priority_nunique = (
    df.groupby("context_hash")["chosen_priority"]
    .nunique()
)

conflict_hashes = set(
    priority_nunique[priority_nunique > 1].index
)

df["priority_conflict"] = df["context_hash"].isin(conflict_hashes)


# Vérifie la structure du model_context.

df["has_context_section"] = df["model_context"].str.contains(
    r"Contexte médical\s*:",
    case=False,
    regex=True,
)

df["has_patient_section"] = df["model_context"].str.contains(
    r"Informations patient\s*:",
    case=False,
    regex=True,
)

df["has_medical_section"] = df["model_context"].str.contains(
    r"Informations médicales\s*:",
    case=False,
    regex=True,
)


# Mesure les longueurs des entrées et sorties.

for col in TEXT_COLS:
    prefix = col.replace("_explanation", "")

    df[f"{prefix}_chars"] = df[col].str.len()
    df[f"{prefix}_words"] = df[col].str.split().str.len()


# Détecte les contextes ressemblant encore à des QCM/questions théoriques.

QCM = re.compile(
    r"\b("
    r"all except|except is|which of the following|true about|false about|"
    r"correct statement|incorrect statement|most appropriate|best answer|"
    r"what is|what are|which is|choose|select|"
    r"toutes sauf|laquelle|lequel|quelle est|quel est|"
    r"vrai concernant|faux concernant"
    r")\b",
    re.I,
)

df["qcm_like"] = df["model_context"].map(
    lambda text: bool(QCM.search(text))
)


# Détecte la présence d'une représentation patient.

PATIENT = re.compile(
    r"\b("
    r"patient|homme|femme|garçon|fille|ans|mois|jours|"
    r"year[- ]old|month[- ]old|day[- ]old|male|female|"
    r"présente|consulte|admis|admitted|presents|presenting|"
    r"history of|antécédent|symptom|symptôme"
    r")\b",
    re.I,
)

df["patient_like"] = df["model_context"].map(
    lambda text: bool(PATIENT.search(text))
)


# Détecte les formulations/artefacts indésirables dans les explications.

FORMULAIC = re.compile(
    r"(?:la priorité est expliquée par|"
    r"cette priorité est justifiée par|"
    r"ceci explique la priorité|"
    r"ces éléments justifient la priorité|"
    r"c['’]est pourquoi la priorité|"
    r"le classement en p[123]|"
    r"the priority is explained by|"
    r"this priority is justified by|"
    r"this explains the priority|"
    r"the chosen response|the rejected response)",
    re.I,
)

MARKDOWN = re.compile(
    r"(?:^|\s)(?:#{1,6}\s|\*\*|__|```|(?:\d+[\.\)])\s)",
    re.M,
)

df["formulaic"] = (
    df["chosen_explanation"].str.contains(FORMULAIC, regex=True)
    | df["rejected_explanation"].str.contains(FORMULAIC, regex=True)
)

df["markdown_artifact"] = (
    df["chosen_explanation"].str.contains(MARKDOWN, regex=True)
    | df["rejected_explanation"].str.contains(MARKDOWN, regex=True)
)


# Compare chosen/rejected pour détecter les paires peu discriminantes.

def explanation_similarity(row: pd.Series) -> float:
    """Calcule la similarité chosen/rejected."""
    chosen = row["chosen_explanation"].strip().lower()
    rejected = row["rejected_explanation"].strip().lower()

    if not chosen or not rejected:
        return 0.0

    return SequenceMatcher(None, chosen, rejected).ratio()


df["explanation_similarity"] = df.apply(
    explanation_similarity,
    axis=1,
)

df["same_explanation"] = (
    df["chosen_explanation"].str.strip().str.lower()
    == df["rejected_explanation"].str.strip().str.lower()
)

df["high_similarity"] = df["explanation_similarity"].ge(0.90)


# Vérifie les langues.

df["language"] = (
    df["language"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["language_invalid"] = ~df["language"].isin(["fr", "en"])


# Sépare erreurs objectives et éléments nécessitant une revue.

CRITICAL_FLAGS = [
    "id_missing",
    "duplicate_id",
    "context_empty",
    "chosen_empty",
    "rejected_empty",
    "chosen_invalid",
    "rejected_invalid",
    "same_priority",
    "hash_invalid",
    "priority_conflict",
    "same_explanation",
]

REVIEW_FLAGS = [
    "qcm_like",
    "formulaic",
    "markdown_artifact",
    "high_similarity",
    "language_invalid",
]

df["critical_issue"] = df[CRITICAL_FLAGS].any(axis=1)
df["needs_review"] = df[REVIEW_FLAGS].any(axis=1)

print()
print(f"Dataset DPO chargé : {len(df):,} lignes")
print(df["source_file"].value_counts().sort_index())

Projet       : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent
Pools triage : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\priority_pools
Splits       : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\splits
Audit        : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\audit

Dataset DPO chargé : 11,830 lignes
source_file
P1    5344
P2    4320
P3    2166
Name: count, dtype: int64


In [20]:
# ============================================================
# AUDIT DPO COMPLET — Rapport + exports
# ============================================================

print("=" * 72)
print("AUDIT GLOBAL DPO")
print("=" * 72)

print(f"\nLignes                  : {len(df):,}")
print(f"IDs uniques             : {df['id'].nunique():,}")
print(f"Contextes uniques       : {df['context_hash'].nunique():,}")
print(f"Colonnes manquantes     : {missing_columns or 'Aucune'}")
print(f"Colonnes supplémentaires: {extra_columns or 'Aucune'}")

print("\n--- SOURCES ---")
print(df["source_file"].value_counts().sort_index().to_string())

print("\n--- LANGUES ---")
print(df["language"].value_counts(dropna=False).to_string())

print("\n--- PRIORITÉS CHOSEN ---")
print(df["chosen_priority"].value_counts().sort_index().to_string())

print("\n--- PRIORITÉS REJECTED ---")
print(df["rejected_priority"].value_counts().sort_index().to_string())

print("\n--- COUPLES CHOSEN → REJECTED ---")
print(
    pd.crosstab(
        df["chosen_priority"],
        df["rejected_priority"],
        rownames=["chosen"],
        colnames=["rejected"],
    ).to_string()
)

print("\n--- STRUCTURE model_context ---")
for col in [
    "has_context_section",
    "has_patient_section",
    "has_medical_section",
    "patient_like",
    "qcm_like",
]:
    print(f"{col:28s}: {df[col].sum():5d} / {len(df)}")

print("\n--- INTÉGRITÉ ---")
FLAGS = CRITICAL_FLAGS + REVIEW_FLAGS + ["duplicate_context"]

for col in FLAGS:
    print(f"{col:28s}: {df[col].sum():5d}")

print("\n--- LONGUEURS ---")
print(
    df[
        [
            "model_context_words",
            "chosen_words",
            "rejected_words",
            "model_context_chars",
            "chosen_chars",
            "rejected_chars"
        ]
    ].describe(percentiles=[.01, .05, .50, .95, .99]).round(1).to_string()
)

print("\n--- SIMILARITÉ CHOSEN / REJECTED ---")
print(
    df["explanation_similarity"]
    .describe(percentiles=[.50, .90, .95, .99])
    .round(3)
    .to_string()
)

print("\n--- CONTEXTES DUPLIQUÉS ---")
dup = df[df["duplicate_context"]]
print(f"Lignes concernées       : {len(dup):,}")
print(f"Contextes concernés      : {dup['context_hash'].nunique():,}")
print(f"Conflits de priorité     : {df['priority_conflict'].sum():,}")

print("\n--- BILAN ---")
print(f"Erreurs critiques        : {df['critical_issue'].sum():,}")
print(f"À revoir                 : {df['needs_review'].sum():,}")
print(
    f"Sans drapeau             : "
    f"{(~df['critical_issue'] & ~df['needs_review']).sum():,}"
)

# ---------- Exports ----------
AUDIT_DIR = Path("../data/audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

df.to_json(
    AUDIT_DIR / "dpo_full_audit.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

df[df["critical_issue"]].to_json(
    AUDIT_DIR / "dpo_critical_issues.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

df[df["needs_review"]].to_json(
    AUDIT_DIR / "dpo_review_queue.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

df[df["priority_conflict"]].to_json(
    AUDIT_DIR / "dpo_priority_conflicts.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print("\n✓ dpo_full_audit.jsonl")
print("✓ dpo_critical_issues.jsonl")
print("✓ dpo_review_queue.jsonl")
print("✓ dpo_priority_conflicts.jsonl")

AUDIT GLOBAL DPO

Lignes                  : 11,830
IDs uniques             : 11,830
Contextes uniques       : 8,560
Colonnes manquantes     : Aucune
Colonnes supplémentaires: Aucune

--- SOURCES ---
source_file
P1    5344
P2    4320
P3    2166

--- LANGUES ---
language
en    11830

--- PRIORITÉS CHOSEN ---
chosen_priority
1     769
2    2017
3    9044

--- PRIORITÉS REJECTED ---
rejected_priority
1     673
2    9793
3    1364

--- COUPLES CHOSEN → REJECTED ---
rejected    1     2     3
chosen                   
1           0   749    20
2         673     0  1344
3           0  9044     0

--- STRUCTURE model_context ---
has_context_section         : 11830 / 11830
has_patient_section         :  8524 / 11830
has_medical_section         :  9734 / 11830
patient_like                :  9632 / 11830
qcm_like                    :  3245 / 11830

--- INTÉGRITÉ ---
id_missing                  :     0
duplicate_id                :     0
context_empty               :     0
chosen_empty             

In [21]:
# ============================================================
# DPO MASTER — Sauvegarde des 11 830 candidats
# ============================================================

MASTER_PATH = OUTPUT_DIR / "dpo_master.jsonl"

FINAL_COLUMNS = [
    "id",
    "language",
    "source_split",
    "model_context",
    "chosen_explanation",
    "chosen_priority",
    "rejected_explanation",
    "rejected_priority",
    "context_hash",
]

df[FINAL_COLUMNS].to_json(
    MASTER_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"DPO master : {len(df):,} lignes")
print(f"✓ {MASTER_PATH}")

DPO master : 11,830 lignes
✓ C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\dpo_master.jsonl


In [23]:
# Score qualité utilisé uniquement pour sélectionner les meilleurs candidats
df["quality_score"] = 0.0

df["quality_score"] += df["has_patient_section"].astype(float) * 2.0
df["quality_score"] += df["has_medical_section"].astype(float) * 1.5
df["quality_score"] += df["patient_like"].astype(float) * 0.5

df["quality_score"] += df["model_context_words"].between(40, 350).astype(float)
df["quality_score"] += df["chosen_words"].between(25, 130).astype(float)
df["quality_score"] += df["rejected_words"].between(25, 130).astype(float)

df["quality_score"] -= df["qcm_like"].astype(float) * 1.5
df["quality_score"] -= df["markdown_artifact"].astype(float) * 1.5
df["quality_score"] -= df["critical_issue"].astype(float) * 10.0

print(df["quality_score"].describe())

count    11830.000000
mean         5.103804
std          1.605467
min         -1.500000
25%          4.000000
50%          5.500000
75%          7.000000
max          7.000000
Name: quality_score, dtype: float64


In [24]:
# ============================================================
# DPO SELECTED — Équilibrage strict sur la plus petite classe
# ============================================================

SELECTED_PATH = OUTPUT_DIR / "dpo_selected.jsonl"

FINAL_COLUMNS = [
    "id",
    "language",
    "source_split",
    "model_context",
    "chosen_explanation",
    "chosen_priority",
    "rejected_explanation",
    "rejected_priority",
    "context_hash",
]


# Distribution disponible dans le master.
counts = (
    df["chosen_priority"]
    .value_counts()
    .reindex([1, 2, 3])
)

if counts.isna().any():
    raise ValueError("Une priorité P1/P2/P3 est absente du DPO master.")

n_per_priority = int(counts.min())

print("Disponibles :")
print(counts.to_string())
print(f"\nSélection par priorité : {n_per_priority:,}")


# Sélectionne les N meilleurs de chaque priorité.
selected_parts = []

for priority in [1, 2, 3]:

    pool = (
        df[df["chosen_priority"].eq(priority)]
        .sort_values(
            ["quality_score", "id"],
            ascending=[False, True],
        )
        .head(n_per_priority)
        .copy()
    )

    selected_parts.append(pool)


selected = pd.concat(
    selected_parts,
    ignore_index=True,
)


# Vérification stricte de l'équilibre.
selected_counts = (
    selected["chosen_priority"]
    .value_counts()
    .reindex([1, 2, 3], fill_value=0)
)

assert selected_counts.nunique() == 1
assert len(selected) == n_per_priority * 3
assert selected["id"].nunique() == len(selected)


# Mélange reproductible.
selected = (
    selected
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)


# Sauvegarde uniquement le schéma DPO final.
selected[FINAL_COLUMNS].to_json(
    SELECTED_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)


# Résumé.
print("\n" + "=" * 60)
print("DPO SELECTED")
print("=" * 60)

print(f"\nLignes             : {len(selected):,}")
print(f"IDs uniques        : {selected['id'].nunique():,}")
print(f"Contextes uniques  : {selected['context_hash'].nunique():,}")

print("\nChosen priorities :")
print(
    selected["chosen_priority"]
    .value_counts()
    .sort_index()
)

print("\nCouples chosen → rejected :")
print(
    pd.crosstab(
        selected["chosen_priority"],
        selected["rejected_priority"],
        rownames=["chosen"],
        colnames=["rejected"],
    )
)

print(f"\n✓ {SELECTED_PATH}")

Disponibles :
chosen_priority
1     769
2    2017
3    9044

Sélection par priorité : 769

DPO SELECTED

Lignes             : 2,307
IDs uniques        : 2,307
Contextes uniques  : 1,874

Chosen priorities :
chosen_priority
1    769
2    769
3    769
Name: count, dtype: int64

Couples chosen → rejected :
rejected    1    2    3
chosen                 
1           0  749   20
2         245    0  524
3           0  769    0

✓ C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\dpo_selected.jsonl


In [25]:
# ============================================================
# DPO SELECTED — Équilibrage strict sur la plus petite classe
# ============================================================

SELECTED_PATH = OUTPUT_DIR / "dpo_selected.jsonl"

FINAL_COLUMNS = [
    "id",
    "language",
    "source_split",
    "model_context",
    "chosen_explanation",
    "chosen_priority",
    "rejected_explanation",
    "rejected_priority",
    "context_hash",
]


# Distribution disponible dans le master.
counts = (
    df["chosen_priority"]
    .value_counts()
    .reindex([1, 2, 3])
)

if counts.isna().any():
    raise ValueError("Une priorité P1/P2/P3 est absente du DPO master.")

n_per_priority = int(counts.min())

print("Disponibles :")
print(counts.to_string())
print(f"\nSélection par priorité : {n_per_priority:,}")


# Sélectionne les N meilleurs de chaque priorité.
selected_parts = []

for priority in [1, 2, 3]:

    pool = (
        df[df["chosen_priority"].eq(priority)]
        .sort_values(
            ["quality_score", "id"],
            ascending=[False, True],
        )
        .head(n_per_priority)
        .copy()
    )

    selected_parts.append(pool)


selected = pd.concat(
    selected_parts,
    ignore_index=True,
)


# Vérification stricte de l'équilibre.
selected_counts = (
    selected["chosen_priority"]
    .value_counts()
    .reindex([1, 2, 3], fill_value=0)
)

assert selected_counts.nunique() == 1
assert len(selected) == n_per_priority * 3
assert selected["id"].nunique() == len(selected)


# Mélange reproductible.
selected = (
    selected
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)


# Sauvegarde uniquement le schéma DPO final.
selected[FINAL_COLUMNS].to_json(
    SELECTED_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)


# Résumé.
print("\n" + "=" * 60)
print("DPO SELECTED")
print("=" * 60)

print(f"\nLignes             : {len(selected):,}")
print(f"IDs uniques        : {selected['id'].nunique():,}")
print(f"Contextes uniques  : {selected['context_hash'].nunique():,}")

print("\nChosen priorities :")
print(
    selected["chosen_priority"]
    .value_counts()
    .sort_index()
)

print("\nCouples chosen → rejected :")
print(
    pd.crosstab(
        selected["chosen_priority"],
        selected["rejected_priority"],
        rownames=["chosen"],
        colnames=["rejected"],
    )
)

print(f"\n✓ {SELECTED_PATH}")

Disponibles :
chosen_priority
1     769
2    2017
3    9044

Sélection par priorité : 769

DPO SELECTED

Lignes             : 2,307
IDs uniques        : 2,307
Contextes uniques  : 1,874

Chosen priorities :
chosen_priority
1    769
2    769
3    769
Name: count, dtype: int64

Couples chosen → rejected :
rejected    1    2    3
chosen                 
1           0  749   20
2         245    0  524
3           0  769    0

✓ C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\dpo_selected.jsonl


In [27]:
# Split DPO groupé par context_hash

import json
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


df = selected.copy()

TRAIN_PATH = RUN_SPLIT_DIR / "dpo_train.jsonl"
VAL_PATH = RUN_SPLIT_DIR / "dpo_validation.jsonl"

SEED = 42
VAL_SIZE = 0.10

assert df["id"].is_unique
assert df["context_hash"].notna().all()
assert df["chosen_priority"].isin([1, 2, 3]).all()


# Recherche du meilleur split sans fuite de contexte

target = (
    df["chosen_priority"]
    .value_counts(normalize=True)
    .reindex([1, 2, 3], fill_value=0)
)

best = None
best_score = float("inf")

for seed in range(SEED, SEED + 100):

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=VAL_SIZE,
        random_state=seed,
    )

    train_idx, val_idx = next(
        splitter.split(df, groups=df["context_hash"])
    )

    train = df.iloc[train_idx]
    val = df.iloc[val_idx]

    train_dist = (
        train["chosen_priority"]
        .value_counts(normalize=True)
        .reindex([1, 2, 3], fill_value=0)
    )

    val_dist = (
        val["chosen_priority"]
        .value_counts(normalize=True)
        .reindex([1, 2, 3], fill_value=0)
    )

    score = (
        (train_dist - target).abs().sum()
        + (val_dist - target).abs().sum()
    )

    if score < best_score:
        best_score = score
        best = train.copy(), val.copy(), seed


train, val, best_seed = best

train = train.sample(frac=1, random_state=SEED).reset_index(drop=True)
val = val.sample(frac=1, random_state=SEED).reset_index(drop=True)


# Vérification anti-fuite

train_hashes = set(train["context_hash"])
val_hashes = set(val["context_hash"])

assert train_hashes.isdisjoint(val_hashes)
assert set(train["id"]).isdisjoint(set(val["id"]))
assert len(train) + len(val) == len(df)


# Sauvegarde

RUN_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train.to_json(
    TRAIN_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

val.to_json(
    VAL_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)


# Résultat

print(f"Seed retenue : {best_seed}")
print(f"Total        : {len(df):,}")
print(f"Train        : {len(train):,} ({len(train)/len(df):.1%})")
print(f"Validation   : {len(val):,} ({len(val)/len(df):.1%})")

print("\nTRAIN")
print(train["chosen_priority"].value_counts().sort_index())

print("\nVALIDATION")
print(val["chosen_priority"].value_counts().sort_index())

print("\nContextes")
print(f"Train      : {train['context_hash'].nunique():,}")
print(f"Validation : {val['context_hash'].nunique():,}")
print(f"Communs    : {len(train_hashes & val_hashes)}")

print("\nFichiers")
print(TRAIN_PATH)
print(VAL_PATH)

Seed retenue : 49
Total        : 2,307
Train        : 2,081 (90.2%)
Validation   : 226 (9.8%)

TRAIN
chosen_priority
1    694
2    694
3    693
Name: count, dtype: int64

VALIDATION
chosen_priority
1    75
2    75
3    76
Name: count, dtype: int64

Contextes
Train      : 1,686
Validation : 188
Communs    : 0

Fichiers
C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\splits\dpo_train.jsonl
C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\dpo_dataset_creation\splits\dpo_validation.jsonl
